# 01 — Data collection and exploration

## 1. Collection context

- **Subject:** anime and users of MyAnimeList (MAL), where users keep their own anime lists and can rate titles 1–10.
- **Unit of observation:** three tables; one row is an anime, a user, and a user–anime pair respectively.
- **Source:** [MyAnimeList Dataset on Kaggle](https://www.kaggle.com/datasets/azathoth42/myanimelist), by Matěj Račinský, 2018, doi: 10.34740/KAGGLE/DSV/45582.
- **License:** Kaggle lists "Database: Open Database, Contents: Database Contents". The author allows use without extra permission or fees and asks for the citation above when the data is used in a publication or presentation.
- **Collection method:** crawl of public data from MyAnimeList.net. MAL identifies users by username, so users cannot be iterated; the author collected usernames from the watching-challenge forum threads of 2015–2018 and from the first ~40k of ~80k clubs.
- **Known limitations:**
  - Users come from forums and clubs, so the sample may lean towards active users and may not represent all MAL users.
  - The data stops around 2018.
  - For each user–anime pair only the current status and the last update are kept, with no history of changes.
- **Version used:** the 3 raw files (`AnimeList.csv`, `UserList.csv`, `UserAnimeList.csv`). The filtered and cleaned versions were processed by the author and are only used for comparison.
- **Why this dataset:** **TODO (team to write)**

**Course requirements:** public, real, traceable, at least ~3,000 rows and 10 columns, not synthetic.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}

print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Number of CSV files:", len(paths))

Python: 3.13.18 | pandas: 2.3.3
Number of CSV files: 9


## 2. Load the data and compare it with the description

**What one row of `UserAnimeList.csv` means:** one user and one anime on that user's list. This step reads only the 4 columns needed for the comparison because the file is about 5 GB.

| Column | Meaning | Data type |
|---|---|---|
| `username` | User identifier | Key |
| `anime_id` | Anime identifier | Key |
| `my_score` | Score given by the user, 1–10 per the description | Ordinal |
| `my_status` | The shelf the user put the anime on | Nominal: the codes are labels, so a mean is meaningless |

`my_status` codes per the description: 1 watching, 2 completed, 3 on hold, 4 dropped, 6 plan to watch; other values are not explained.

Before writing `assert`s we print the actual numbers next to the description's numbers, because some differences may be legitimate.

In [2]:
# Safe load: only an empty cell is missing. pandas would also read the username `null` as missing
na_kwargs = dict(keep_default_na=False, na_values=[""])
anime_raw = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
users_raw = pd.read_csv(paths["UserList.csv"], **na_kwargs)
lists_raw = pd.read_csv(
    paths["UserAnimeList.csv"], **na_kwargs,
    usecols=["username", "anime_id", "my_score", "my_status"],
    dtype={"username": "category", "anime_id": "int32",
           "my_score": "int32", "my_status": "int32"},
)

# Compare with the description first, no assert yet
checks = {
    "records":       (len(lists_raw), 80_076_112),
    "records rated": (int((lists_raw.my_score > 0).sum()), 46_358_322),
    "users (lists)": (lists_raw.username.nunique(), 302_675),
    "users (table)": (users_raw.username.nunique(), 302_675),
    "anime (table)": (anime_raw.anime_id.nunique(), 14_478),
    "anime (lists)": (lists_raw.anime_id.nunique(), 14_478),
}
for name, (actual, expected) in checks.items():
    print(f"{name:15s} actual={actual:>12,} description={expected:>12,} diff={actual - expected:+,}")

print("anime_id unique:", anime_raw.anime_id.is_unique)
print("title unique   :", anime_raw.title.is_unique)
print(lists_raw.my_status.value_counts().sort_index())
print(lists_raw.my_score.value_counts().sort_index())

records         actual=  80,076,112 description=  80,076,112 diff=+0
records rated   actual=  46,358,322 description=  46,358,322 diff=+0
users (lists)   actual=     283,045 description=     302,675 diff=-19,630
users (table)   actual=     302,675 description=     302,675 diff=+0
anime (table)   actual=      14,478 description=      14,478 diff=+0
anime (lists)   actual=      14,478 description=      14,478 diff=+0
anime_id unique: True
title unique   : False
my_status
0          527
1      4057788
2     50445526
3      2985804
4      3194353
5            6
6     19392104
33           3
55           1
Name: count, dtype: int64
my_score
0     33717790
1       321270
2       374744
3       619587
4      1296354
5      2905183
6      5456518
7     10379184
8     11394082
9      7897871
10     5713529
Name: count, dtype: int64


**Where this step sits.** Step 1 of the exploration order (see [docs/exploration_order.md](../docs/exploration_order.md)). This record holds the **second** run of notebook 01, with the safe load (only an empty cell is missing). The first run, which used pandas defaults and lost the username `null`, is kept in the git history and compared with this one in 2.2 c. The statistics of 2.2 a (scores by status) are results on raw data and are re-checked after cleaning.

### 2.1 Comparison with the description

**What we do here.** The author of the dataset wrote a description: how many records each file has, which columns exist, and which codes a column may hold. We do not take it on trust. For each claim we count the same thing in the real files and write down every place where the file says something different. *Matches* means we found the same number. *Differs* means we found another number.

**How the files are read.** Only an **empty cell** counts as missing (`keep_default_na=False, na_values=[""]`). Pandas would also read words such as `null` as missing, and that would lose a real user (see 2.2 c).

**Matches.**
- 80,076,112 records in the list table.
- **46,358,322 records with a rating.**
- 14,478 anime in both the anime table and the list table, and the two sets of `anime_id` are identical.
- **302,675 usernames in the user table, with no repeated username.**

**Differs from the description.**

| Item | Description | Actual | Difference | Explanation | Status |
|---|---|---|---|---|---|
| Usernames in `UserAnimeList` | 302,675 | 283,045 | −19,630 | Users with an empty list own no row (see 2.2 d) | Explained for 19,622 of 19,630; 8 users are not |
| `my_status` values | 1, 2, 3, 4, 6 | also 0, 5, 33, 55 | 537 rows | The description says other values are unknown | Decision needed (see 2.2 b) |

How to read the differences:
- **Usernames in the list table.** The description says 302,675 users, but only 283,045 users have at least one row. The other 19,630 users exist in the user table and own no row in the list table. 19,622 of them declare an empty list, so nothing is lost. 8 declare anime but have no rows, and we do not know why (see 2.2 d).
- **`my_status` values.** 537 rows out of 80 million carry a code that the description does not define. We cannot know what they mean, so we only report them (see 2.2 b).

**How `my_status` is stored.** Each row of the list table is one user and one anime, and `my_status` says where that anime sits on the user's list. The file stores a **number**, not a word. The words in the table below come from the author's description. *Share* is the number of rows with that code divided by 80,076,112.

| `my_status` | Meaning | Rows | Share |
|---|---|---|---|
| 2 | completed | 50,445,526 | 63.0% |
| 6 | plan to watch | 19,392,104 | 24.2% |
| 1 | watching | 4,057,788 | 5.1% |
| 4 | dropped | 3,194,353 | 4.0% |
| 3 | on hold | 2,985,804 | 3.7% |
| 0, 5, 33, 55 | unknown | 537 | under 0.001% |

Most rows (63.0%) are completed anime. About a quarter (24.2%) are only plans: the user has not watched them yet. Only 12.8% are anime that someone is watching, has dropped or has put on hold.

**How `my_score` is stored.** The description says scores go from 1 to 10, but the file also contains 0. The number of rows with a score of 1 or more is exactly 46,358,322, the number of "records with a rating" in the description. So **0 does not mean "the user gave 0 points". It means "no score given".** There are 33,717,790 such rows (42.1%).

- **Why this matters (pay attention to this).** A 0 must become a missing value before any mean is computed. Example: a user scored two anime 8 and 10 and left three unrated. With the zeros the mean is (8 + 10 + 0 + 0 + 0) / 5 = 3.6. The correct mean of the real scores is 9.
- **Score 0 does not mean "watched but did not rate"**, because plan to watch usually means not watched yet. The split by `my_status` is in 2.2 a.
- **Among the rated rows the mean is about 7.49, and the scores lean high.** 76.3% of the rated rows are 7 or more, only 5.6% are 4 or less, and the most common score is 8 (11,394,082 rows). People mostly score what they like, so scores are not spread evenly between 1 and 10.
- **`title` is not unique** (2 anime share a title), so `anime_id` is the key (see 2.2 e).

In [3]:
# 1) Turn what matched the description into asserts
assert len(lists_raw) == 80_076_112
assert (lists_raw.my_score > 0).sum() == 46_358_322
assert lists_raw.my_score.between(0, 10).all()
assert anime_raw.anime_id.nunique() == 14_478
assert lists_raw.anime_id.nunique() == 14_478
assert anime_raw.anime_id.is_unique

# 2) For each status: how many rows are unrated, and what is the mean score?
print(lists_raw.groupby("my_status").my_score.agg(
    rows="size",
    unrated=lambda s: (s == 0).sum(),
    pct_unrated=lambda s: round((s == 0).mean() * 100, 1),
    mean_rated=lambda s: s[s > 0].mean(),
))

# 3) Rows whose status is not in the description
known_status = [1, 2, 3, 4, 6]
odd_status = lists_raw[~lists_raw.my_status.isin(known_status)]
print("\nodd status rows:", len(odd_status), "| users:", odd_status.username.nunique())
print(odd_status.head(8))

# 4) Missing username: re-read as raw text to see the real name
username_text = pd.read_csv(paths["UserList.csv"], usecols=["username"],
                            dtype=str, keep_default_na=False)["username"]
print("\nusers_raw rows:", len(users_raw),
      "| username NaN:", users_raw.username.isna().sum(),
      "| matching raw text:", username_text[users_raw.username.isna()].tolist(),
      "| duplicated usernames:", users_raw.username.dropna().duplicated().sum())
print("lists username NaN:", lists_raw.username.isna().sum())

# 5) Users with no row in the list table: empty list or missing crawl?
list_users = set(lists_raw.username.cat.categories)
table_users = set(users_raw.username.dropna())
print("in lists, not in table:", len(list_users - table_users))
count_cols = ["user_watching", "user_completed", "user_onhold",
              "user_dropped", "user_plantowatch"]
users_no_list = users_raw[~users_raw.username.isin(list_users)]
declared = users_no_list[count_cols].sum(axis=1)
print("users without list:", len(users_no_list),
      "| declare 0 anime:", (declared == 0).sum(),
      "| declare > 0:", (declared > 0).sum())
print(declared[declared > 0].describe())

# 6) Key of the list table; do the two anime id sets really match?
print("\nduplicated (username, anime_id):",
      lists_raw.duplicated(["username", "anime_id"]).sum())
print("anime_id mismatch between tables:",
      len(set(lists_raw.anime_id) ^ set(anime_raw.anime_id)))

# 7) Anime sharing a title
same_title = anime_raw[anime_raw.title.duplicated(keep=False)]
print("\nanime with a duplicated title:", len(same_title))
print(same_title.sort_values("title")[
    ["anime_id", "title", "type", "aired_string"]].head(10))

               rows   unrated  pct_unrated  mean_rated
my_status                                             
0               527       450         85.4    9.000000
1           4057788   2784664         68.6    7.952967
2          50445526   8046553         16.0    7.573541
3           2985804   2044638         68.5    7.267161
4           3194353   1687539         52.8    4.948901
5                 6         5         83.3    7.000000
6          19392104  19153938         98.8    7.852393
33                3         2         66.7    5.000000
55                1         1        100.0         NaN

odd status rows: 537 | users: 24
               username  anime_id  my_score  my_status
4452617       hatnclogs        21        10          0
4452623       hatnclogs       223        10          0
4452627       hatnclogs       249        10          0
4452644       hatnclogs      1604        10          0
4452726       hatnclogs     11061        10          0
4452790       hatnclogs     220

### 2.2 Conclusions from the run (EDA first stage)

The numbers below come from the output of the check cell above, run on Kaggle with the raw data and the safe load. Each point says what we counted, what the number means, and what we do not know.

**a) Score 0 and status: who really has not rated?**

How to read the table: *Not rated* is the number of rows with score 0 in that status group. *% not rated* is that number divided by the rows of the group. *Mean score* is computed only over the rows that have a score.

| `my_status` | Meaning | Rows | Not rated | % not rated | Mean score (rated rows only) |
|---|---|---|---|---|---|
| 2 | completed | 50,445,526 | 8,046,553 | 16.0% | 7.57 |
| 1 | watching | 4,057,788 | 2,784,664 | 68.6% | 7.95 |
| 3 | on hold | 2,985,804 | 2,044,638 | 68.5% | 7.27 |
| 4 | dropped | 3,194,353 | 1,687,539 | 52.8% | 4.95 |
| 6 | plan to watch | 19,392,104 | 19,153,938 | 98.8% | 7.85 |

- **Most unrated rows are plans, not forgotten scores.** There are 33,717,790 unrated rows in total, and plan to watch accounts for 56.8% of them. This is natural: nobody scores an anime they have not watched (98.8% of plan-to-watch rows are unrated). Without plan to watch, the unrated share of the remaining rows is 24.0% (14,563,852 of 60,684,008), not 42.1%.
- **The mean score over all rated rows is 7.49.**
- **People who dropped an anime score it much lower than people who completed it:** 4.95 against 7.57, a gap of 2.62 points. This only describes the data. Nothing is controlled, so we do not know the cause.
- **Only 47.2% of dropped rows have a score.** The 4.95 comes only from the people who chose to rate after dropping, and they may not be like everyone who dropped. Keep this in mind for any question about anime quality.
- **A small oddity.** 238,166 plan-to-watch rows (1.2% of that group) still have a score, with a mean of 7.85. The user may have changed the status after rating, but the data cannot tell, so we only note it.
- **This table is the same in both runs** (the first with pandas defaults, the second with the safe load). The safe load only fixed the usernames. This table counts rows by status and never uses the username, and the 261 rows of the user `null` are inside it in both runs (for example 158 of the 50,445,526 completed rows).

**b) 537 rows with a status outside the description (0, 5, 33, 55)**

| Code | Rows | Not rated | Mean score of the rated rows |
|---|---|---|---|
| 0 | 527 | 450 (85.4%) | 9.00 (77 rated rows) |
| 5 | 6 | 5 (83.3%) | 7.00 |
| 33 | 3 | 2 (66.7%) | 5.00 |
| 55 | 1 | 1 (100%) | no score |

- **They come from only 24 users**, so they are not widespread, and 98% of the rows carry the code 0. For example `hatnclogs` has several status-0 rows with a score of 10, and `programmargorp` has status-0 rows with no score.
- **The mean of 9.00 for code 0 is not reliable.** It rests on 77 rows, and the rows printed in the output show one user (`hatnclogs`) giving a 10 several times.
- 537 rows are under 0.001% of 80,076,112 rows, so the effect on any statistic is close to zero.
- **We do not know what these codes mean.** The description says other values are unknown.
- **Proposal (the team decides):** exclude them from analyses that use `my_status` and report how many rows were removed.

**c) The username `null`: one real user that pandas read as "missing"**

**Terms used in this section.**
- **User table** = `UserList.csv`. One row is one user. It has a `username` column, and columns with the size of the user's list as the user declares it on the profile (watching, completed, on hold, dropped, plan to watch).
- **List table** = `UserAnimeList.csv`. One row is one anime on one user's list. The same username appears in as many rows as the user has anime.
- The two tables are joined on `username`.

**1. What happened.** There is **exactly one user whose username is the word `null`**. In the user table it is **one row**, because `username` is unique there (302,675 rows, 302,675 different usernames). In the list table the same user has **261 rows**, one per anime.

By default `pd.read_csv` turns the words `null`, `NA`, `N/A`, `NaN`, `None` and the empty cell into a missing value. This is the *normal load*. Under the normal load pandas read the username `null` as a missing value, so:
- in the user table, this user became "a row without a username";
- in the list table, its 261 rows became "261 rows without a username";
- the two sides could not be joined, because a missing value matches nothing.

**2. The fix, called the safe load.** We read every file with `keep_default_na=False, na_values=[""]`. This switches off the default list of words, so only an **empty cell** is missing, and the word `null` stays as text. Words such as `None` and `Unknown` also stay as text, so we decide ourselves in step 3 which of them mean "missing".

**3. The two runs of this notebook side by side.** The first run used the normal load, the second run the safe load.

| | First run (normal load) | Second run (safe load) |
|---|---|---|
| Usernames in the user table | 302,674 | **302,675** (matches the description) |
| Missing usernames in the user table | 1 | **0** |
| List rows with a missing username | 261 | **0** |
| Users with at least one list row | 283,044 | 283,045 |
| Users without a list row | 19,631 | 19,630 |
| Of those, users who declare more than 0 anime | 9 | 8 |

**What the last row means.** It counts users who exist in the user table and declare anime on their profile, but own **no row in the list table**. It is a group of users defined by comparing the two tables, not a table of its own. In the first run the user `null` was in this group, only because its 261 rows had lost their username and could not be matched. In the second run the rows are matched, so the user leaves the group: 9 becomes 8. The other numbers moved for the same reason: one user was found, so users with rows go up by one and users without rows go down by one.

**4. Two different 261.** The number 261 appears in both tables, but it counts different things:
- **User table:** one row, the user `null`, with a declared list size of **261 anime**.
- **List table:** **261 rows** with the username `null`, one per anime.

They agree: the profile says 261 anime and the list table holds 261 rows. There are not 261 users called `null`.

**5. What we verified (from the Kaggle output).**
- The check that looks for a missing username in `UserList.csv` printed `[]`. No missing username is left.
- `username` is unique in the user table (`True`). `user_id` is **not** unique (302,673 different values for 302,675 rows), so we never join on `user_id`.
- The 261 rows of the user `null` carry normal statuses (for example 158 are status 2, completed). They are ordinary data and are not part of the 537 rows of section b.
- The table of section a does not use the username, so it is the same in both runs.

**6. What we have NOT verified yet.** The agreement 261 = 261 compares two counts for one user. It is good evidence that the rows belong to the user `null`, but it does not prove that the join works for every user. Two checks are still open, both in step 3 (`kaggle/03_raw_quality_audit.ipynb`, not yet run):
- every username of the list table exists in the user table, and the other way round (no orphan on either side);
- for every user, the declared list size equals the number of rows in the list table.

Until they run, we can say that the user `null` is explained, but not that the join on `username` is perfect.

**Rule from now on:** every notebook reads the tables with `keep_default_na=False, na_values=[""]`.

**d) 19,630 users with no row in the list table**

- 19,622 of them (99.96%) declare 0 anime in `UserList`. They have **empty lists**, so this is not missing crawl data.
- **8 users declare anime but have no rows.** Their declared list sizes are 1 at the smallest, 146 in the middle (median) and 1,071 at the largest, with a mean of 289. In the first run there were 9: the count fell to 8 once `null` was read correctly, so the ninth was the user `null`, who does own rows. For the 8 that remain we do not know the reason (a list that could not be crawled, a private list, or a user who emptied the list after the profile was crawled). Step 3 checks whether these users have other activity values.
- Consequence: an inner merge (keep only users found in both tables) drops this group naturally. It does not remove any row of the list table.

**e) Keys and identifiers**

A *key* is a column (or a pair of columns) whose value identifies exactly one row.
- There is **no repeated `(username, anime_id)` pair** (confirmed again on all 80,076,112 rows). So one row of the list table is exactly one user and one anime.
- The two `anime_id` sets (anime table and list table) are identical (0 ids differ).
- `title` is repeated once: `Hinamatsuri` is anime 35102 (an OVA) and also anime 36296 (a TV series that started airing on 6 April 2018). They are different anime, so `anime_id` stays the key and `title` is never used to join.

**What this means for the research questions (hypotheses, not verified):**
- **Questions about dropping:** compute on users who started watching (status 1, 2, 3, 4) and leave plan to watch out of the denominator.
- **Questions about scores:** say clearly that scores only reflect people who chose to rate, and that the share of rated rows differs a lot between status groups.

## 3. Data quality exploration

**Where the evidence comes from.** The audit is done in [kaggle/02_column_dictionary_run.ipynb](../kaggle/02_column_dictionary_run.ipynb) (formats, placeholders, missing values per column) and [kaggle/03_raw_quality_audit.ipynb](../kaggle/03_raw_quality_audit.ipynb) (placeholders, impossible values, inconsistencies between tables). This step only measures. The order of all steps is in [docs/exploration_order.md](../docs/exploration_order.md).

**TODO:** Document duplicates / empty rows, numerical distributions / implausible values / outliers, categorical frequencies / inconsistent labels, missingness patterns and relationships relevant to candidate questions. Explain why each issue matters, not merely print outputs.

In [4]:
# Examples; choose only meaningful checks once data is loaded.
# display(missing_overview(raw_df))
# print("Duplicate rows:", raw_df.duplicated().sum())
# display(raw_df.describe(include="all").T)


## 4. Cleaning decisions, then save processed dataset

**TODO:** For each cleaning step, explain the rule and why it is justified. Use `report_row_change(step, before_df, after_df, reason)` for filtering, dropping and merging. Preserve original raw data. Do not assume all missing values or outliers should be removed.

**Open decisions** (impossible values, unknown status codes, placeholders, not-yet-aired anime, missing demographics, users without a list) are listed in [docs/exploration_order.md](../docs/exploration_order.md). After this step, the statistics of step 5 ([kaggle/05a_verify_meanings.ipynb](../kaggle/05a_verify_meanings.ipynb) and the two next to it) are re-run on the processed data.

In [5]:
# Example pattern only; replace with a justified operation.
# before_df = raw_df.copy()
# processed_df = before_df.copy()  # TODO: actual cleaning rule
# report_row_change("Step name", before_df, processed_df, "Explain the operation")
# processed_df.to_csv(ROOT / "data/processed/cleaned.csv", index=False)


## 5. Exploration summary

Write **3–5 evidence-based observations**, serious quality problems, a preprocessing plan and possible directions for research questions. Keep observation separate from interpretation. (Team must write this section themselves.)